# Bedrock を API Gateway + AWS Lambda から呼び出す

## 環境のセットアップ

APIGW - Lambda - Bedrock の構成です。環境は SAM テンプレートで定義しています。

`sam deploy --config-file bedrock-apigw.toml` などでデプロイしてください。

## 呼び出し

Converse API は、`https://<APIGW>/Prod/converse?text=こんにちは` で呼び出せます

ConverseStream API は、`https://<APIGW>/Prod/stream?text=こんにちは` で呼び出せます

## 解説 (Converse)

特別な注意点はありません。Converse API を Lambda から実行して、それをレスポンスに含めれば OK です。

実装例（Python）

```python
import json
import boto3

bedrock_runtime = boto3.client('bedrock-runtime')
modelId = "jp.amazon.nova-2-lite-v1:0"

def handler(event, context):
    params = event.get('queryStringParameters') or {}
    text = params.get('text', 'こんにちは')
    
    response = bedrock_runtime.converse(
        modelId=modelId,
        messages = [{
          "role": "user",
          "content": [{"text": text}],
        }],
        inferenceConfig = {
          "maxTokens": 2048,
          "temperature": 0
        }
    )
    
    return {
        'statusCode': 200,
        'body': response["output"]["message"]["content"][0]["text"]
    }
```

## 解説 (ConverseStream)

Lambda からストリーム形式でレスポンスを返すときは、[レスポンスストリーミング機能](https://docs.aws.amazon.com/ja_jp/lambda/latest/dg/configuration-response-streaming.html) を使用します。これは、Lambda の Function URL 機能を使うか、もしくは InvokeWithResponseStream API を呼び出すことで利用できます。今回は、API Gateway から呼び出したいので、後者の方法を採用します。また、Node.js の場合はデフォルトでレスポンスストリーミングに対応していますが、他のランタイムの場合は対応していないので、カスタムランタイムを作成するか Lambda Web Adapter を使用する必要があります。今回は Node.js を採用します。なお、サンプルコートが [AWS 公式 GitHub](https://github.com/aws-samples/serverless-samples/tree/main/apigw-response-streaming) に掲載されています。

API Gateway とこの Lambda を統合する場合、[いくつかの注意点](https://docs.aws.amazon.com/ja_jp/apigateway/latest/developerguide/response-transfer-mode-lambda.html) があります。まず、統合時に、レスポンス転送モードを `STREAM` に設定のうえ、InvokeWithResponseStream 形式で Lambda 関数を呼び出すようにします。そのうえで、指定のフォーマットでレスポンスが返るように Lambda 関数の実装を調整します。

関連する API Gateway の設定を定義するテンプレートは以下の箇所です。なお、この記述をシンプルにしてほしいという [リクエスト](https://github.com/aws/serverless-application-model/pull/3881) が存在しています。

```yaml
  StreamMethod:
    Type: AWS::ApiGateway::Method
    Properties:
      # 略
      Integration:
        Type: AWS_PROXY
        IntegrationHttpMethod: POST
        Uri: !Sub 'arn:aws:apigateway:${AWS::Region}:lambda:path/2021-11-15/functions/${BedrockStreamFunction.Arn}/response-streaming-invocations'
        ResponseTransferMode: STREAM
```

Lambda 関数 (Node.js) のサンプル実装

```javascript
const { BedrockRuntimeClient, ConverseStreamCommand } = require("@aws-sdk/client-bedrock-runtime");

const client = new BedrockRuntimeClient();
const modelId = "jp.amazon.nova-2-lite-v1:0";

exports.handler = awslambda.streamifyResponse(async (event, responseStream) => {
  const metadata = {
    statusCode: 200,
    headers: {
      "Content-Type": "text/plain; charset=utf-8"
    }
  };
  
  responseStream = awslambda.HttpResponseStream.from(responseStream, metadata);
  
  try {
    const text = event.queryStringParameters?.text || "こんにちは";
    
    const command = new ConverseStreamCommand({
      modelId,
      messages: [{ role: "user", content: [{ text }] }],
      inferenceConfig: { maxTokens: 2048, temperature: 0 }
    });
    
    const response = await client.send(command);
    
    for await (const chunk of response.stream) {
      if (chunk.contentBlockDelta?.delta?.text) {
        responseStream.write(chunk.contentBlockDelta.delta.text);
      }
    }
  } catch (error) {
    responseStream.write(`Error: ${error.message}`);
  }
  
  responseStream.end();
});
```

## クリーンアップ

`sam delete --config-file bedrock-apigw.toml` などでクリーンアップしてください。